# Benchmark: Axisymmetric cooling and spreading

In [ ]:
import subprocess
import os
import xarray as xr
import rasterio as rio
from rasterio.enums import Resampling
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import sys
import math
import glob
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor

#### This benchmark is a temperature-dependent test on a viscous flow. 

In this experiment, a hot Newtonian fluid is extruded from a point source onto a horizontal plane and allowed to cool to the ambient air temperature. The propogation of such a flow has an analytical solution, which is plotted alongside the models.

We have the initial volume set in accordance with the [Garel et al., 2012](https://appliedvolc.biomedcentral.com/articles/10.1186/s13617-017-0061-x#ref-CR26) experiment.

We encourage changing the volume betweens runs to compare the physical or stochastic reactions at various scales.

In [ ]:
volume = 3.432e-6

#### The following cells should not be edited, unless you have extensive knowledge of the model.

The following cell sets up common parameters. Following this, the next 4 cells format and generate input files for each respective model, at parameters considered reasonable for this benchmark.

In [ ]:
easting = 0
northing = 0
dem = "DEMs_Lava_Benchmark/topography_dem_axi.asc"
raster = rxr.open_rasterio(dem)
raster = raster.sel({"band": 1})
xll, yll = float(raster.x.min()), float(raster.y.min())
coordinates = np.array([int(easting),int(northing)])
r = rio.open(dem)
res = r.res
run_time = 156
ratio = volume/3.432e-6

output_num = str(int(run_time/100))
out = output_num.zfill(4)

### Molasses: Additional Notes

Molasses input parameters are stored in `custom_molasses.conf` in the `Molasses` folder. The most impactful change would be that of changing the minimum and maximum residual, as the cellular automata structure will cause a much larger or smaller area to be impacted. As a stochastic model, no physical parameters are used.

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/Molasses')
f = open("custom_molasses.conf","r+")
conf = f.readlines()
conf[2] = f"""MIN_RESIDUAL = {4e-5*ratio}\n"""
conf[3] = f"""MAX_RESIDUAL = {1e-3*ratio}\n"""
conf[4] = f"""MIN_TOTAL_VOLUME = {str(volume)}\n"""
conf[5] = f"""MAX_TOTAL_VOLUME = {str(volume)}\n"""
conf[6] = f"""MIN_PULSE_VOLUME = {float(volume)/1000}\n"""
conf[7] = f"""MAX_PULSE_VOLUME = {float(volume)/100}\n"""
conf[10] = f"""DEM_FILE = ../{dem}\n"""
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

f=open("events.in","w")
f.write(f"""{easting},{northing}""")
f.close()

subprocess.run("molasses custom_molasses.conf",shell=True,stderr=subprocess.DEVNULL)

victor.convert_molasses("molasses_axisymmetric",resolution=res)

### MrLavaLoba/Flowy: Additional Notes

MrLavaLoba/Flowy (a drop in C++ replacement) input parameters are stored in `input.toml` in the `flowy` folder. The most impactful changes would be those such as the thickening parameter, lobe exponent, or max slope probability. Other parameters not mentioned will also affect the creation and placement of lobes. As a stochastic model, no physical parameters are specified by the user.

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/MrLavaLoba/')
g=open("input_data.py","r+")
inp = g.readlines()
inp[1] = """run_name = 'Axisymmetric'\n"""
inp[3] = f"""source = '../{dem}'\n"""
inp[41] = """vent_flag = 0\n"""
inp[43] = f"""x_vent = [0]\n"""
inp[44] = f"""y_vent = [0]\n"""
inp[74] = f"""n_flows = 16\n"""
inp[79] = f"""min_n_lobes = 1000\n"""
inp[80] = f"""max_n_lobes = 1000\n"""
inp[98] = f"""lobe_area = 1.16e-5\n"""
inp[108] = f"""thickness_ratio = 1\n"""
inp[118] = f"""thickening_parameter = 0.1\n"""
inp[130] = f"""lobe_exponent = 1\n"""
inp[140] = f"""max_slope_prob = 0.5\n"""
inp[148] = f"""inertial_exponent = 0.1\n"""
inp[90] = f"""total_volume = {volume}\n"""
g.seek(0)
g.writelines(inp)
g.truncate()
g.close()

subprocess.run("python mr_lava_loba.py",shell=True)

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/IMEX_LavaFlow')
radius = 2e-3/ratio
subprocess.run("mv IMEX_LavaFlow_bm3.inp IMEX_LavaFlow.inp", shell=True)
h=open("IMEX_LavaFlow.inp","r+")
inp = h.readlines()
inp[100] = f""" VEL_SOURCE= {1.8e-3*ratio}\n"""
h.seek(0)
h.writelines(inp)
h.truncate()
h.close()

p = subprocess.Popen(['./IMEX_LavaFlow'], stdin=subprocess.PIPE, shell=True)
p.communicate(input=b'\n')
subprocess.run("mv IMEX_LavaFlow.inp IMEX_LavaFlow_bm3.inp", shell=True)

In [ ]:
scale = ratio
os.chdir('/home/jovyan/Lava Benchmark/Lava2d')

subprocess.run("python example_BM3_axisymmetric.py", shell=True)
subprocess.run("python convert_output.py outputs/out.nc -o lava2d_axisymmetric.asc", shell=True)

#### The final two cells concern visualization.

The first provides a reference to the final timestamp of each simulation, though all outputs are avilable in their models' respective folder. The second and last cell plots each model side by side. We encourage users to edit this to their personal preference.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/")

mll_outputs = glob.glob(f'mr_lava_loba/Axisymmetric*thickness_masked*.asc')
mll_out = mll_outputs[-1]

r = rxr.open_rasterio("IMEX_LavaFlow/BM3_Axisymmetric_max.asc",masked=True)
r2 = rxr.open_rasterio("Molasses/molasses_axisymmetric.asc",masked=True)
r3 = rxr.open_rasterio(mll_out,masked=True)
r4 = rxr.open_rasterio("lava2d/lava2d_axisymmetric.asc",masked=True)
r_min, r_max = r.min(), r.max()
r2_min, r2_max = r2.min(), r2.max()
r3_min, r3_max = r3.min(), r3.max()
r4_min, r4_max = r4.min()/10000, r4.max()/10000

maximum = np.max((r_max,r2_max,r3_max,r4_max))
idx_max = np.argmax((r_max,r2_max,r3_max,r4_max))

In [ ]:
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2, figsize = (12,9))

thickness0 = victor.plot_flow(dem, "Molasses/molasses_axisymmetric.asc", axes=ax0,zoom=False,title='Molasses',minimum=1e-6,scale=maximum)
thickness1 = victor.plot_flow(dem, mll_out, axes=ax1, zoom=False,title='MrLavaLoba',minimum=1e-6, scale=maximum)
thickness2 = victor.plot_flow(dem, "IMEX_LavaFlow/BM3_Axisymmetric_max.asc", axes=ax2, zoom=False,title='IMEX_LavaFlow', minimum=1e-6, scale=maximum)
thickness3 = victor.plot_flow(dem, "lava2d/lava2d_axisymmetric.asc", zoom=False, axes=ax3 ,title='Lava2D', minimum=1e-6, scale=maximum*1000)

cbar_ax = fig.add_axes([.25, 0, 0.5, 0.05])
thickness = [thickness0, thickness1, thickness2, thickness3]
thickness = thickness[idx_max]
fig.colorbar(thickness, cax=cbar_ax, label="Flow Thickness (m)", orientation="horizontal")